# Bundle Adjustment — API Demo

How to apply bundle adjustment (BA) to a feedforward pointcloud result.

**Prerequisites:** [Keyframe Extraction](../01_preprocessing/keyframe_extraction.ipynb) → [Feedforward Methods](feedforward_methods.ipynb)

**Not an evaluation.** For benchmark results across `baseline / ba / lc`, see `eval_7scenes_gt.ipynb`. Compute lives in `evals/eval_gt.py`.

Two API styles are demonstrated:

- **Section A**: full pipeline — `creator.run(IMAGES)` → `ba.refine(result)` → `creator.reproject(refined)` → `creator.build_colmap(RECON)`
- **Section B**: cached result path — load `FeedforwardResult` from zarr, call `ba.refine()` without re-running GPU inference (no reproject — `raw_outputs` not persisted)

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import shutil
from pathlib import Path

import torch
import zarr

from collab_splats.pointcloud import BundleAdjustment, BundleAdjustmentConfig
from collab_splats.pointcloud.feedforward import VGGTXCreator
from collab_splats.pointcloud.feedforward.base import FeedforwardResult

## §0 — Setup

Source the common tutorial configuration, then assert that prerequisite notebooks have run.
All paths derive from `CACHE_DIR` set by `tutorial_config.py`.

In [ ]:
%run ../tutorial_config.py

# ── Configuration ─────────────────────────────────────────────────────────────
METHOD    = "vggtx"    # "vggtx" | "mapanything"
VARIANT   = "ba"       # "ba" | "lc" | "" (empty = raw baseline)

VGGTX_ZARR = CACHE_DIR / "vggtx" / "reconstruction.zarr"
RECON      = CACHE_DIR / METHOD / VARIANT if VARIANT else CACHE_DIR / METHOD
RECON.mkdir(parents=True, exist_ok=True)

In [4]:
assert IMAGES.exists() and any(IMAGES.iterdir()), (
    f"No images in {IMAGES}. Run 01_preprocessing/keyframe_extraction first."
)
assert VGGTX_ZARR.exists(), (
    f"Feedforward zarr not found at {VGGTX_ZARR}. Run 02_pointcloud/feedforward_methods first."
)

## §1 — Section A: Full Pipeline

`creator.run()` runs the feedforward model and returns a `FeedforwardResult`.
`BundleAdjustment.refine()` refines poses in-place (returns a new `FeedforwardResult` with updated extrinsics/intrinsics).
`creator.reproject()` re-extracts world points using the refined poses.
`creator.build_colmap()` exports the result to COLMAP format.

In [ ]:
creator = VGGTXCreator()
ba = BundleAdjustment(config=BundleAdjustmentConfig())

# Run feedforward inference; loads model + builds FeedforwardResult with raw_outputs
ff_result = creator.run(IMAGES)
print(f"feedforward: {ff_result.pts3d.shape[0]:,} points, {ff_result.extrinsics.shape[0]} cameras")

# Refine camera poses — returns new FeedforwardResult with updated extrinsics/intrinsics
refined = ba.refine(ff_result)

# Re-project world points using refined poses (requires raw_outputs from creator.run())
reprojected = creator.reproject(refined)
print(f"after BA: {reprojected.pts3d.shape[0]:,} points")

# Export to COLMAP format
colmap_result = creator.build_colmap(RECON)
print(f"COLMAP: {len(colmap_result.points):,} points, {colmap_result.camera_poses.shape[0]} cameras")

## §2 — Section B: Cached Result Path (no GPU re-run)

Load a pre-computed `FeedforwardResult` from the zarr written by `feedforward_methods.ipynb`,
then call `ba.refine()` without re-loading the model.

> **Note:** `creator.reproject()` is not available in this path — `raw_outputs` from the
> original inference run is not persisted to zarr. `pts3d` in the refined result reflects
> the original feedforward geometry, not re-projected points.

In [ ]:
# Load cached feedforward result; restore images from zarr store (needed for track extraction)
ff_result = FeedforwardResult.load_zarr(VGGTX_ZARR)
_store = zarr.open(str(VGGTX_ZARR), mode="r")
ff_result.images = torch.from_numpy(_store["images"][:])   # (N, 3, H, W)

# Save pre-BA extrinsics for the visualisation below
extrinsics_pre = ff_result.extrinsics.copy()   # (N, 4, 4)

# Refine poses — enable loss history capture so we can plot the convergence curve
ba = BundleAdjustment(config=BundleAdjustmentConfig(capture_loss_history=True))
refined = ba.refine(ff_result)
print(f"refined: {refined.extrinsics.shape[0]} cameras  pts3d: {refined.pts3d.shape[0]:,} points")

## §3 — Visualizations

Pre/post BA camera positions and LM loss convergence curve.

`extrinsics_pre` and `refined.extrinsics` are both `(N, 4, 4)` world-to-camera matrices.
Camera center in world coordinates is `−R.T @ t`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 — registers 3D projection


def _camera_centers(extrinsics: np.ndarray) -> np.ndarray:
    """Return world-space camera centers from (N, 4, 4) world-to-camera extrinsics."""
    R = extrinsics[:, :3, :3]   # (N, 3, 3) rotation
    t = extrinsics[:, :3, 3]    # (N, 3) translation
    # Camera center in world = -R.T @ t (inverse of world-to-camera translation)
    return np.einsum("nij,nj->ni", R.transpose(0, 2, 1), -t)


# Extract per-camera world positions before and after BA
centers_pre  = _camera_centers(extrinsics_pre)
centers_post = _camera_centers(refined.extrinsics)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

# Trajectory arcs — connect consecutive cameras with a line
ax.plot(*centers_pre.T,  "o-", color="steelblue",  label="Pre-BA",  alpha=0.8, markersize=5)
ax.plot(*centers_post.T, "o-", color="darkorange", label="Post-BA", alpha=0.8, markersize=5)

# Per-camera displacement connectors — show how much each camera moved
for pre, post in zip(centers_pre, centers_post):
    ax.plot(
        [pre[0], post[0]], [pre[1], post[1]], [pre[2], post[2]],
        color="gray", alpha=0.35, linewidth=0.8, linestyle="--",
    )

ax.set_xlabel("X (m)")
ax.set_ylabel("Y (m)")
ax.set_zlabel("Z (m)")
ax.legend(fontsize=11)
ax.set_title("Camera Positions: Pre vs Post BA")
plt.tight_layout()
plt.show()

# Summary: mean and max displacement per camera
deltas = np.linalg.norm(centers_post - centers_pre, axis=-1)
print(f"Camera displacement — mean: {deltas.mean():.4f} m  max: {deltas.max():.4f} m")

In [ ]:
import matplotlib.pyplot as plt


loss_hist = ba._last_loss_history

if not loss_hist:
    print("No loss history captured. Re-run with BundleAdjustmentConfig(capture_loss_history=True).")
else:
    fig, ax = plt.subplots(figsize=(8, 4))
    steps = list(range(1, len(loss_hist) + 1))

    # Loss curve on log scale — LM losses can span several orders of magnitude
    ax.semilogy(steps, loss_hist, "o-", color="steelblue", markersize=5, linewidth=1.5)

    # Final-loss reference line
    ax.axhline(
        loss_hist[-1], color="gray", linestyle="--", alpha=0.6,
        label=f"Final: {loss_hist[-1]:.3e}",
    )

    ax.set_xlabel("LM Step")
    ax.set_ylabel("Loss (log scale)")
    ax.set_title("Bundle Adjustment — LM Loss Convergence")
    ax.legend(fontsize=11)
    ax.grid(True, which="both", alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Print improvement ratio
    print(
        f"Loss reduction: {loss_hist[0]:.3e} → {loss_hist[-1]:.3e}  "
        f"({loss_hist[0] / loss_hist[-1]:.1f}× improvement)"
    )

## Where to go next

- **Benchmarks across baseline/BA/LC** — `evals/eval_gt.py` (compute) + `docs/pointcloud/eval_7scenes_gt.ipynb` (viz)
- **Loop closure path** — `collab_splats.pointcloud.LoopClosure`
- **Tunable knobs** — `BundleAdjustmentConfig` (`max_reproj_error`, `lm_steps`, `shared_camera`, `min_inliers_per_frame`, `device`)
- **Source** — `collab_splats/pointcloud/bundle_adjustment.py` (BundleAdjustment class, BundleAdjustmentConfig), `collab_splats/pointcloud/feedforward/base.py` (BaseFeedforwardCreator.run, .reproject)